# BootstrapFinetune (Chapter 6)

This notebook accompanies Chapter 6 §6.6 of *Context Engineering with DSPy*.

> **Requires a GPU and a running Ollama server.** This notebook fine-tunes a local model (default: `Qwen/Qwen2.5-0.5B-Instruct`) and then serves the resulting checkpoint via [Ollama](https://ollama.com/download) for inference. On Apple Silicon use the companion `finetune-mac-m3.ipynb` notebook (MPS path). On CUDA you can run this notebook end-to-end.

Set `TRAINSET_LIMIT` and `TESTSET_LIMIT` env vars to small numbers (e.g. `5`) for a fast smoke test before committing to a full run.


## Setup

Install the required dependencies. The fine-tuning packages are the same across all platforms.

In [ ]:
# Core dependencies (all platforms)
%pip install -r ../requirements.txt -q

# Ollama runs as a separate local service:
# https://ollama.com/download

In [ ]:
import dspy
import torch
import random
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables (explicit path avoids notebook/frame detection issues).
load_dotenv(dotenv_path=Path.cwd() / ".env")

# Disable W&B auto-init for non-interactive/local runs.
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Set up logging to see fine-tuning progress
logging.basicConfig(level=logging.INFO)

# Patch LocalProvider.kill() so BootstrapFinetune can call lm.kill() safely
# when no local inference server process is running.
from dspy.clients.lm_local import LocalProvider

_original_kill = LocalProvider.kill.__func__ if hasattr(LocalProvider.kill, '__func__') else LocalProvider.kill

@staticmethod
def _patched_kill(lm, launch_kwargs=None):
    if not hasattr(lm, "process"):
        logging.getLogger(__name__).info("No running local server to kill.")
        return
    _original_kill(lm, launch_kwargs)

LocalProvider.kill = _patched_kill

print(f"DSPy version: {dspy.__version__}")
print(f"PyTorch version: {torch.__version__}")


# Compatibility shim for TRL API change:
# newer TRL versions renamed `max_seq_length` -> `max_length`.
import inspect
from trl import SFTConfig as _TRL_SFTConfig

if "max_seq_length" not in inspect.signature(_TRL_SFTConfig).parameters:
    _original_sft_init = _TRL_SFTConfig.__init__

    def _compat_sft_init(self, *args, max_seq_length=None, **kwargs):
        if max_seq_length is not None and "max_length" not in kwargs:
            kwargs["max_length"] = max_seq_length
        _original_sft_init(self, *args, **kwargs)

    _TRL_SFTConfig.__init__ = _compat_sft_init


## Device Detection

DSPy's training code automatically detects the best available device. Let's verify what's available on your system.

In [ ]:
def get_device_info():
    """Detect available compute device for local training."""
    if torch.cuda.is_available():
        device = "cuda"
        device_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ CUDA GPU available: {device_name}")
        print(f"   VRAM: {vram_gb:.1f} GB")
        print(f"   CUDA version: {torch.version.cuda}")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = "mps"
        print("✅ Apple Silicon GPU available (MPS)")
        # Test MPS
        try:
            x = torch.ones(1, device="mps")
            print("   MPS test: PASSED")
        except Exception as e:
            print(f"   MPS test: FAILED - {e}")
    else:
        device = "cpu"
        print("⚠️ No GPU available - using CPU (training will be slow)")

    return device

DEVICE = get_device_info()


## Load Dataset

We'll use the same AI vs Human text detection task from the optimizer benchmark. This is a binary classification task where we need to determine if text was written by an AI or a human.

In [ ]:
import pandas as pd

# Load the dataset
csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')

# Convert to DSPy Examples
dataset = [
    dspy.Example(**ex).with_inputs("text")
    for ex in examples
]

# Reproducible shuffle and split
random.seed(42)
random.shuffle(dataset)

n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)

trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

# Optional limits for faster local smoke tests.
train_limit = int(os.getenv("TRAINSET_LIMIT", "0"))
test_limit = int(os.getenv("TESTSET_LIMIT", "0"))
if train_limit > 0:
    trainset = trainset[:train_limit]
if test_limit > 0:
    testset = testset[:test_limit]

print(f"Dataset: {csv_path}")
print(f"Training set:   {len(trainset)} examples")
print(f"Validation set: {len(valset)} examples")
print(f"Test set:       {len(testset)} examples")
print(f"\nExample:")
print(f"  Text: {trainset[0].text[:100]}...")
print(f"  Label: {trainset[0].is_ai}")


## Define DSPy Module

We define a simple Chain-of-Thought classifier that reasons about whether text is AI-generated.

In [ ]:
class DetectAIText(dspy.Signature):
    """Analyze text and determine if it was written by an AI or a human.
    
    Look for patterns like:
    - Overly formal or generic language
    - Lack of personal voice or unique perspective
    - Perfect grammar without natural speech patterns
    - Hedging language ("It's important to note...", "In conclusion...")
    """
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="True if AI-generated, False if human-written")


class AIDetector(dspy.Module):
    """A Chain-of-Thought classifier for AI text detection."""
    
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


# Metric for evaluation
def exact_match(example, prediction, trace=None):
    """Check if the prediction matches the ground truth label."""
    return 1 if example.is_ai == prediction.is_ai else 0


print("Module defined: AIDetector")
print("Metric defined: exact_match")

## Configure Models

We'll set up:
1. **Teacher model**: A capable API model (GPT-4o-mini) that generates high-quality reasoning traces
2. **Student model**: A small local model (Qwen2.5-0.5B) that we'll fine-tune

The teacher generates successful traces on the training set, which become the training data for the student.

In [ ]:
import requests

# Model configuration
STUDENT_MODEL = os.getenv("STUDENT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
TEACHER_MODEL = os.getenv("TEACHER_MODEL", "llama3.2:latest")
OLLAMA_API_BASE = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")

# Teacher LM (Ollama, local and cross-platform)
try:
    health = requests.get(f"{OLLAMA_API_BASE}/api/tags", timeout=5)
    health.raise_for_status()
except Exception as exc:
    raise RuntimeError(
        f"Could not reach Ollama at {OLLAMA_API_BASE}. Start Ollama and try again."
    ) from exc

teacher_lm = dspy.LM(
    f"ollama_chat/{TEACHER_MODEL}",
    api_base=OLLAMA_API_BASE,
    api_key="",
    temperature=0.2,
    max_tokens=1000,
)

print(f"Teacher model (Ollama): {TEACHER_MODEL}")
print(f"Student model: {STUDENT_MODEL}")
print(f"Ollama endpoint: {OLLAMA_API_BASE}")
print(f"Training device: {DEVICE}")


## Set Up Student Model

For the student model, we use DSPy's `LocalProvider` which handles local model fine-tuning.

**Important**: The student model must have an LM assigned before fine-tuning.

In [ ]:
from dspy.clients.lm_local import LocalProvider

# Create local provider for fine-tuning
local_provider = LocalProvider()

# Create student LM
student_lm = dspy.LM(
    model=f"openai/local:{STUDENT_MODEL}",
    provider=local_provider,
    max_tokens=500,
)

# Create teacher and student modules
teacher_module = AIDetector()
teacher_module.set_lm(teacher_lm)

student_module = AIDetector()
student_module.set_lm(student_lm)

print("Teacher module configured with:", TEACHER_MODEL)
print("Student module configured with:", STUDENT_MODEL)

## Run BootstrapFinetune

Now we run the BootstrapFinetune optimizer. Here's what happens:

1. **Bootstrapping**: The teacher module runs on each training example, generating reasoning traces
2. **Filtering**: If a metric is provided, only successful traces (where metric returns True) are kept
3. **Data Preparation**: Traces are formatted into training data for the student model
4. **Fine-tuning**: The student model is fine-tuned on the bootstrapped data using HuggingFace/TRL

### Training Configuration

You can customize training with `train_kwargs`:

In [ ]:
# Training hyperparameters
num_train_epochs = int(os.getenv("NUM_TRAIN_EPOCHS", "3"))

train_kwargs = {
    "num_train_epochs": num_train_epochs,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5,
    "bf16": DEVICE == "cuda",  # Use bf16 only on CUDA
    "use_peft": True,  # Use LoRA for efficient fine-tuning
    "report_to": "none",  # Disable external experiment trackers
    "disable_tqdm": True,  # Avoid notebook progress callback issues in headless runs
}

print("Training configuration:")
for k, v in train_kwargs.items():
    print(f"  {k}: {v}")


In [ ]:
from dspy.teleprompt import BootstrapFinetune

# Optional: Set output directory for fine-tuned model
os.environ["DSPY_FINETUNEDIR"] = str(Path.cwd() / "finetuned_models")
bootstrap_num_threads = int(os.getenv("BOOTSTRAP_NUM_THREADS", "1"))

print("Starting BootstrapFinetune...")
print(f"Training examples: {len(trainset)}")
print(f"Output directory: {os.environ['DSPY_FINETUNEDIR']}")
print(f"Bootstrap threads: {bootstrap_num_threads}")
print()

# Create optimizer with metric
optimizer = BootstrapFinetune(
    metric=exact_match,  # Filter traces where prediction matches label
    num_threads=bootstrap_num_threads,
    train_kwargs=train_kwargs,
)

# Run fine-tuning
# This will:
# 1. Run teacher on trainset, collecting successful traces
# 2. Format traces as training data
# 3. Fine-tune the student model
finetuned_module = optimizer.compile(
    student=student_module,
    teacher=teacher_module,
    trainset=trainset,
)

print("\n" + "="*50)
print("Fine-tuning complete!")
print("="*50)


## Running Inference (Ollama)

This notebook uses Ollama for inference on all platforms.

Before running the next cells:
1. Deploy your model to Ollama (if needed, merge adapter + base model and convert to GGUF first).
2. Start Ollama locally.
3. Set `OLLAMA_MODEL` (and optionally `OLLAMA_API_BASE`).


In [ ]:
print("Inference runtime: Ollama (cross-platform)")
print("Set OLLAMA_MODEL before running the next cell.")


### Configure Ollama Endpoint

Environment variables used below:

- `OLLAMA_MODEL`: required model tag in Ollama (for example, `ai-detector-finetuned`)
- `OLLAMA_API_BASE`: optional, defaults to `http://localhost:11434`


In [ ]:
import requests

ollama_model = os.getenv("OLLAMA_MODEL")
ollama_api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")

# Helpful context for debugging deployment issues.
finetuned_model_path = finetuned_module.get_lm().model
if finetuned_model_path.startswith("openai/local:"):
    finetuned_model_path = finetuned_model_path[len("openai/local:"):]

if not ollama_model:
    raise ValueError(
        "Missing OLLAMA_MODEL. Set OLLAMA_MODEL to your deployed Ollama model name "
        "(for example: export OLLAMA_MODEL=ai-detector-finetuned). "
        "BootstrapFinetune produced a local checkpoint at: "
        + finetuned_model_path
    )

# Check Ollama health before binding DSPy to it.
try:
    health = requests.get(f"{ollama_api_base}/api/tags", timeout=5)
    health.raise_for_status()
except Exception as exc:
    raise RuntimeError(
        f"Could not reach Ollama at {ollama_api_base}. Start Ollama and try again."
    ) from exc

print(f"Connecting to Ollama model: {ollama_model}")
print(f"Ollama endpoint: {ollama_api_base}")

finetuned_lm = dspy.LM(
    f"ollama_chat/{ollama_model}",
    api_base=ollama_api_base,
    api_key="",
    max_tokens=500,
    temperature=0.0,
)
finetuned_module.set_lm(finetuned_lm)

# Test inference through DSPy
test_text = testset[0].text
result = finetuned_module(text=test_text)

print(f"\nTest prediction:")
print(f"  Text: {test_text[:100]}...")
print(f"  Predicted: {result.is_ai}")
print(f"  Actual: {testset[0].is_ai}")
print(f"  Reasoning: {result.reasoning[:200]}...")


## Evaluate Fine-tuned Model

Let's evaluate the fine-tuned model on the test set and compare with the teacher model.

In [ ]:
eval_num_threads = int(os.getenv("EVAL_NUM_THREADS", "1"))

# Optional limit to speed up local checks.
evalset = testset
eval_limit = int(os.getenv("EVAL_LIMIT", "0"))
if eval_limit > 0:
    evalset = testset[:eval_limit]

def _score_value(result):
    return float(result.score) if hasattr(result, "score") else float(result)

evaluator = dspy.Evaluate(
    devset=evalset,
    metric=exact_match,
    num_threads=eval_num_threads,
    display_progress=True,
    display_table=False,
)

print("Evaluating fine-tuned model...")
finetuned_result = evaluator(finetuned_module)
finetuned_score = _score_value(finetuned_result)
print(f"Fine-tuned accuracy: {finetuned_score:.1f}%")

print("\nEvaluating teacher model...")
teacher_result = evaluator(teacher_module)
teacher_score = _score_value(teacher_result)
print(f"Teacher accuracy: {teacher_score:.1f}%")


## Cleanup

No local inference server is started by this notebook in Ollama mode.


In [ ]:
print("No cleanup required for Ollama mode.")


## Summary

In this notebook, we demonstrated:

1. **BootstrapFinetune workflow**: Using a teacher model to generate successful traces, then fine-tuning a smaller student model on those traces.

2. **Cross-platform training**: DSPy's training works on CUDA, MPS (Mac), and CPU.

3. **Single cross-platform inference path**: Use Ollama (`ollama_chat/...`) on Mac, Windows, and Linux.

### Key Takeaways

- Fine-tuning distills the reasoning capabilities of a large model into a smaller one
- The `metric` parameter filters training data to only include successful traces
- `train_kwargs` lets you customize training hyperparameters (epochs, batch size, LoRA, etc.)
- A single Ollama deployment path keeps demos consistent across machines

### Next Steps

- Try with a larger student model for better performance
- Experiment with different `train_kwargs` settings
- Use BootstrapFinetune without a metric for unsupervised distillation
- Automate adapter merge + GGUF conversion in your deployment pipeline
